# Agent IA — Authentification Continue et Adaptive
### Pipeline complet : Chargement → Preprocessing → Modèle ML → Scoring → Décision

## 1. Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay

# Charger les fichiers
train_transaction = pd.read_csv(r'C:\Users\Pc\Desktop\TER\ieee-fraud-detection\train_transaction.csv')
train_identity    = pd.read_csv(r'C:\Users\Pc\Desktop\TER\ieee-fraud-detection\train_identity.csv')

print('Transactions:', train_transaction.shape)
print('Identity:',     train_identity.shape)

## 2. Fusion et filtrage

In [ ]:
# Fusionner et garder seulement les lignes avec infos d'appareil
df = train_transaction.merge(train_identity, on='TransactionID', how='left')
df_complet = df[df['DeviceType'].notna()].copy()

print('Dataset filtré:', df_complet.shape)
print('Fraudes:', df_complet['isFraud'].value_counts().to_dict())
print(f'Taux de fraude : {df_complet["isFraud"].mean()*100:.2f}%')

## 3. Preprocessing — Encodage + Split
> **Important** : les `label_encoders` sont sauvegardés ici pour être réutilisés dans le scoring.

In [ ]:
colonnes = ['TransactionAmt', 'card1', 'DeviceType', 'DeviceInfo', 'id_30', 'id_31', 'isFraud']
df_model = df_complet[colonnes].copy()

# Gestion des valeurs manquantes
df_model['TransactionAmt'] = df_model['TransactionAmt'].fillna(0)
df_model['card1']          = df_model['card1'].fillna(0)

# Encodage + sauvegarde de chaque encodeur
label_encoders = {}
for col in ['DeviceType', 'DeviceInfo', 'id_30', 'id_31']:
    df_model[col] = df_model[col].fillna('inconnu')
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    label_encoders[col] = le  # on garde chaque encodeur

# Split train / test
X = df_model.drop('isFraud', axis=1)
y = df_model['isFraud']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape, '| Fraudes train:', y_train.sum())
print('Test: ', X_test.shape,  '| Fraudes test: ', y_test.sum())

## 4. Entraînement Random Forest (Class Weight)
> **Note** : Random Forest n'a pas besoin de StandardScaler. Le scaler n'est donc pas utilisé ici.

In [ ]:
clf_final = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
clf_final.fit(X_train, y_train)

# Évaluation sur X_test brut (pas de scaler)
y_pred  = clf_final.predict(X_test)
y_proba = clf_final.predict_proba(X_test)[:, 1]

print('=== ÉVALUATION — Random Forest + Class Weight ===')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraude']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_proba):.4f}')

In [ ]:
# Visualisation ROC + Matrice de confusion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[0], name='RF + Class Weight')
axes[0].plot([0,1], [0,1], 'k--', alpha=0.5)
axes[0].set_title('Courbe ROC')

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Normal', 'Fraude'],
    ax=axes[1], colorbar=False
)
axes[1].set_title('Matrice de confusion')

plt.tight_layout()
plt.show()

In [ ]:
# Importance des features
features = ['TransactionAmt', 'card1', 'DeviceType', 'DeviceInfo', 'id_30', 'id_31']
df_imp = pd.DataFrame({'Feature': features, 'Importance': clf_final.feature_importances_})
df_imp = df_imp.sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 4))
plt.barh(df_imp['Feature'], df_imp['Importance'], color='#2E5F9E')
plt.xlabel('Importance')
plt.title('Importance des signaux dans le modèle')
plt.tight_layout()
plt.show()

print(df_imp.to_string(index=False))

## 5. Construction des profils comportementaux

In [ ]:
colonnes_profil = ['TransactionAmt', 'card1', 'DeviceType', 'id_30', 'id_31', 'TransactionDT']
df_profil = df_complet[colonnes_profil].copy()
df_profil['heure'] = (df_profil['TransactionDT'] // 3600) % 24

profils = df_profil.groupby('card1').agg(
    appareil_habituel  = ('DeviceType', lambda x: x.mode()[0] if len(x.mode()) > 0 else 'inconnu'),
    os_habituel        = ('id_30',      lambda x: x.mode()[0] if len(x.mode()) > 0 else 'inconnu'),
    navigateur_habituel= ('id_31',      lambda x: x.mode()[0] if len(x.mode()) > 0 else 'inconnu'),
    montant_moyen      = ('TransactionAmt', 'mean'),
    montant_std        = ('TransactionAmt', 'std'),
    heure_moyenne      = ('heure', 'mean'),
    heure_std          = ('heure', 'std'),
    nb_transactions    = ('TransactionAmt', 'count'),
).reset_index()

profils['montant_std'] = profils['montant_std'].fillna(50)
profils['heure_std']   = profils['heure_std'].fillna(6)

print(f'Profils construits : {len(profils)}')
print(profils.head(3))

## 6. Moteur de scoring comportemental

In [ ]:
def calculer_score(transaction, profil):
    """
    Compare une transaction au profil habituel de la carte.
    Retourne un score entre 0 (très suspect) et 1 (normal).
    """
    scores = {}

    scores['appareil'] = 1.0 if transaction['DeviceType'] == profil['appareil_habituel'] else 0.0
    scores['os']       = 1.0 if transaction['id_30']      == profil['os_habituel']       else 0.0

    z_montant      = abs(transaction['TransactionAmt'] - profil['montant_moyen']) / (profil['montant_std'] + 1)
    scores['montant'] = max(0.0, 1.0 - z_montant / 3)

    heure          = (transaction['TransactionDT'] // 3600) % 24
    z_heure        = abs(heure - profil['heure_moyenne']) / (profil['heure_std'] + 1)
    scores['heure']   = max(0.0, 1.0 - z_heure / 3)

    score_global = (
        0.30 * scores['appareil'] +
        0.25 * scores['os']      +
        0.25 * scores['montant'] +
        0.20 * scores['heure']
    )
    return round(score_global, 4), scores


def decision_agent(score):
    if score >= 0.8:
        return 'VERT',   'Accès accordé — session normale',                          '✅'
    elif score >= 0.5:
        return 'ORANGE', 'Vérification requise — code OTP envoyé',                   '⚠️'
    else:
        return 'ROUGE',  'Accès bloqué — alerte générée, nouveau code requis',       '🚨'

print('Fonctions de scoring et décision définies ✅')

## 7. Score combiné ML + comportemental

In [ ]:
def score_combine(transaction_row, profils, modele, label_encoders, poids_ml=0.6, poids_scoring=0.4):
    """
    Combine le score ML (Random Forest) et le score comportemental.
    Les deux scores mesurent la même chose : 1 = normal, 0 = suspect.
    
    Note : pas de scaler — Random Forest travaille sur les valeurs brutes encodées.
    """
    # Étape 1 : Encoder les colonnes catégoriques
    cat_cols = ['DeviceType', 'DeviceInfo', 'id_30', 'id_31']
    cat_encoded = []
    for col in cat_cols:
        le  = label_encoders[col]
        val = transaction_row[col]
        if val in le.classes_:
            encoded = le.transform([val])[0]
        else:
            encoded = -1  # valeur inconnue = signal d'anomalie
        cat_encoded.append(encoded)

    # Étape 2 : Construire le vecteur dans le même ordre qu'à l'entraînement
    # Ordre : TransactionAmt, card1, DeviceType, DeviceInfo, id_30, id_31
    card1_val = transaction_row['card1']
    X_input = [[
        transaction_row['TransactionAmt'],
        card1_val,
        cat_encoded[0],  # DeviceType
        cat_encoded[1],  # DeviceInfo
        cat_encoded[2],  # id_30
        cat_encoded[3],  # id_31
    ]]

    # Étape 3 : Score ML — probabilité de fraude inversée
    proba_fraude = modele.predict_proba(X_input)[0][1]
    score_ml = 1 - proba_fraude  # 1 = normal, 0 = suspect

    # Étape 4 : Score comportemental
    profil_carte = profils[profils['card1'] == card1_val]
    if len(profil_carte) == 0:
        score_comp = 0.5  # carte inconnue → score neutre
    else:
        score_comp, _ = calculer_score(transaction_row, profil_carte.iloc[0])

    # Étape 5 : Score final combiné
    score_final = poids_ml * score_ml + poids_scoring * score_comp

    return round(score_final, 4), round(score_ml, 4), round(score_comp, 4)

print('Fonction score_combine définie ✅')

## 8. Test du pipeline complet

In [ ]:
# Test sur 5 transactions réelles du dataset
colonnes_test = ['TransactionAmt', 'card1', 'DeviceType', 'DeviceInfo', 'id_30', 'id_31', 'TransactionDT']
test_sample = df_complet[colonnes_test].sample(5, random_state=42)

print('=== TEST PIPELINE COMPLET ===')
print()
for _, row in test_sample.iterrows():
    s_final, s_ml, s_comp = score_combine(row, profils, clf_final, label_encoders)
    niveau, action, emoji = decision_agent(s_final)
    print(f'Score ML={s_ml} | Score Comportemental={s_comp} | Score Final={s_final} → {niveau} {emoji}')
    print(f'  Action : {action}')
    print()

In [ ]:
# Test sur une transaction suspecte simulée
carte_test   = profils[profils['nb_transactions'] >= 5]['card1'].iloc[0]
profil_test  = profils[profils['card1'] == carte_test].iloc[0]

transaction_normale = {
    'card1':          carte_test,
    'DeviceType':     profil_test['appareil_habituel'],
    'DeviceInfo':     df_complet[df_complet['card1'] == carte_test]['DeviceInfo'].mode()[0],
    'id_30':          profil_test['os_habituel'],
    'id_31':          profil_test['navigateur_habituel'],
    'TransactionAmt': profil_test['montant_moyen'],
    'TransactionDT':  int(profil_test['heure_moyenne']) * 3600
}

transaction_suspecte = {
    'card1':          carte_test,
    'DeviceType':     'mobile' if profil_test['appareil_habituel'] == 'desktop' else 'desktop',
    'DeviceInfo':     'unknown_device',
    'id_30':          'Android 9.0',
    'id_31':          'unknown_browser',
    'TransactionAmt': profil_test['montant_moyen'] * 15,
    'TransactionDT':  3 * 3600  # 3h du matin
}

print('=== SIMULATION ===')
for nom, tx in [('NORMALE', transaction_normale), ('SUSPECTE', transaction_suspecte)]:
    s_final, s_ml, s_comp = score_combine(tx, profils, clf_final, label_encoders)
    niveau, action, emoji = decision_agent(s_final)
    print(f'Transaction {nom}')
    print(f'  Score ML={s_ml} | Score Comportemental={s_comp} | Score Final={s_final}')
    print(f'  Décision : {niveau} {emoji} — {action}')
    print()

## 9. Sauvegarde de tous les artefacts

In [ ]:
# Modèle
with open('model_final.pkl', 'wb') as f:
    pickle.dump(clf_final, f)

# Encodeurs
with open('label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

# Profils comportementaux
profils.to_csv('profils_utilisateurs.csv', index=False)
with open('profils_utilisateurs.pkl', 'wb') as f:
    pickle.dump(profils, f)

print('Sauvegardés ✅')
print('  model_final.pkl')
print('  label_encoders.pkl')
print('  profils_utilisateurs.csv')
print('  profils_utilisateurs.pkl')